In [1]:
#%pip install scikit-learn pandas numpy xgboost
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

Importing the data n stuff

In [2]:
pipes = pd.read_csv("Water_Pipes_Feb10.csv", encoding='cp1252', dtype={10: str})
breaks = pd.read_csv("Breaks2013toNow.csv")

In [3]:
breaks["break_date"] = pd.to_datetime(breaks["break_date"], errors="coerce")

In [4]:
pipes.isna().sum()

OBJECTID               0
tag                    0
status                 0
cbu                    0
owner                  0
metered                0
fire_line              0
diameter               0
material               0
polywrap               0
yr_inst                0
length                 0
yr_abandon             0
yr_removed             0
yr_replace             0
comments               0
NumberOfMa             0
Shape_Leng             0
Score                  0
Soil_Comp              9
Soil_Comp_Full         9
Soil_Hydro_Group    1051
Shape_Length           0
ElevChange             4
NumberofSLs            0
dtype: int64

In [5]:
pipes['Soil_Hydro_Group'] = pipes['Soil_Hydro_Group'].fillna("No Group")

In [6]:
pipes.isna().sum()

OBJECTID            0
tag                 0
status              0
cbu                 0
owner               0
metered             0
fire_line           0
diameter            0
material            0
polywrap            0
yr_inst             0
length              0
yr_abandon          0
yr_removed          0
yr_replace          0
comments            0
NumberOfMa          0
Shape_Leng          0
Score               0
Soil_Comp           9
Soil_Comp_Full      9
Soil_Hydro_Group    0
Shape_Length        0
ElevChange          4
NumberofSLs         0
dtype: int64

In [7]:
pipes = pipes.dropna()

In [8]:
pipes.isna().sum()

OBJECTID            0
tag                 0
status              0
cbu                 0
owner               0
metered             0
fire_line           0
diameter            0
material            0
polywrap            0
yr_inst             0
length              0
yr_abandon          0
yr_removed          0
yr_replace          0
comments            0
NumberOfMa          0
Shape_Leng          0
Score               0
Soil_Comp           0
Soil_Comp_Full      0
Soil_Hydro_Group    0
Shape_Length        0
ElevChange          0
NumberofSLs         0
dtype: int64

In [9]:
def clean_polywrap(x):
    if x == "n":
        return 0.0
    try:
        return float(x)
    except:
        return np.nan


pipes["polywrap"] = pipes["polywrap"].apply(clean_polywrap)

In [10]:
pipes["yr_inst"] = pd.to_numeric(pipes["yr_inst"], errors="coerce")

In [11]:
pipes.head()

,OBJECTID,tag,status,cbu,owner,metered,fire_line,diameter,material,polywrap,...,comments,NumberOfMa,Shape_Leng,Score,Soil_Comp,Soil_Comp_Full,Soil_Hydro_Group,Shape_Length,ElevChange,NumberofSLs
0,1,m411,i,y,CBU,n,n,6.0,CI,0.0,...,,3,425.246940,2,CtC,"Crider-Urban land complex, 6 to 12 percent slopes",Group B,425.246940,1.648987,9
1,2,m4117,i,y,CBU,n,n,2.0,C,0.0,...,,0,57.600633,0,CtC,"Crider-Urban land complex, 6 to 12 percent slopes",Group B,57.600633,0.349121,0
2,3,m4130,i,y,CBU,n,n,12.0,DI,0.0,...,,0,28.431573,4,CtC,"Crider-Urban land complex, 6 to 12 percent slopes",Group B,28.431573,0.013092,0
3,4,m4032,i,y,CBU,n,n,6.0,CI,0.0,...,,0,18.126310,3,Ua,"Udorthents, loamy",No Group,18.126310,0.582748,1
4,5,m4046,i,y,CBU,n,n,6.0,CI,0.0,...,,0,90.664972,3,Ua,"Udorthents, loamy",No Group,90.664972,1.354767,1


Monthly snapshot creation

In [12]:
print(breaks['break_date'].min(), breaks['break_date'].max())

2013-01-01 00:00:00 2025-08-15 00:00:00


In [13]:
SNAPSHOT_START = "2013-01-01"
SNAPSHOT_END   = "2025-09-01"

snapshots = pd.date_range(
    SNAPSHOT_START,
    SNAPSHOT_END,
    freq="MS"
)

In [14]:
rows = []

for _, pipe in pipes.iterrows():

    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    if pd.isna(install_year):
        continue

    pipe_breaks = breaks.loc[
        breaks["pipe_tag"] == tag,
        "break_date"
    ].sort_values()

    for snap in snapshots:

        if snap.year < install_year:
            continue

        past = pipe_breaks[pipe_breaks < snap]

        if len(past) == 0:
            months_since_last = np.nan
            breaks_so_far = 0
        else:
            last_break = past.max()
            months_since_last = (snap - last_break).days / 30.44
            breaks_so_far = len(past)

        rows.append({
            "tag": tag,
            "snapshot": snap,

            "diameter": pipe["diameter"],
            "material": pipe["material"],
            "polywrap": pipe["polywrap"],
            "yr_inst": pipe["yr_inst"],
            "length": pipe["length"],
            "NumberOfMa": pipe["NumberOfMa"],
            "Shape_Length": pipe["Shape_Length"],
            "Score": pipe["Score"],
            "Soil_Comp": pipe["Soil_Comp"],
            "Soil_Hydro_Group": pipe["Soil_Hydro_Group"],
            "ElevChange": pipe["ElevChange"],
            "NumberofSLs": pipe["NumberofSLs"],
            "age_years": snap.year - pipe["yr_inst"],
            "breaks_so_far": breaks_so_far,
            "months_since_last": months_since_last
        })

snap_df = pd.DataFrame(rows)

In [15]:
snap_df.head()

,tag,snapshot,diameter,material,polywrap,yr_inst,length,NumberOfMa,Shape_Length,Score,Soil_Comp,Soil_Hydro_Group,ElevChange,NumberofSLs,age_years,breaks_so_far,months_since_last
0,m411,2013-01-01,6.0,CI,0.0,1961,425,3,425.24694,2,CtC,Group B,1.648987,9,52,0,NaN
1,m411,2013-02-01,6.0,CI,0.0,1961,425,3,425.24694,2,CtC,Group B,1.648987,9,52,0,NaN
2,m411,2013-03-01,6.0,CI,0.0,1961,425,3,425.24694,2,CtC,Group B,1.648987,9,52,0,NaN
3,m411,2013-04-01,6.0,CI,0.0,1961,425,3,425.24694,2,CtC,Group B,1.648987,9,52,0,NaN
4,m411,2013-05-01,6.0,CI,0.0,1961,425,3,425.24694,2,CtC,Group B,1.648987,9,52,0,NaN


preprocessesing stuff

In [16]:
breaks_by_tag = (
    breaks
    .dropna(subset=["pipe_tag", "break_date"])
    .sort_values("break_date")
    .groupby("pipe_tag")["break_date"]
    .apply(np.array)
)

def broke_in_next(tag, snap, months):
    if tag not in breaks_by_tag:
        return 0

    dates = breaks_by_tag[tag]
    start = snap
    end   = snap + pd.DateOffset(months=months)
    start = np.datetime64(start)
    end   = np.datetime64(end)
    i = np.searchsorted(dates, start, side="right")

    if i >= len(dates):
        return 0

    return int(dates[i] <= end)

X = 24 #months
    
snap_df["Y"] = snap_df.apply(
    lambda r: broke_in_next(r["tag"], r["snapshot"], X),
    axis=1
)

In [ ]:
feature_cols = [
    "diameter", #c
    "material", #c
    "polywrap",
    "yr_inst", #c
    "length",
    "NumberOfMa",
    "Shape_Length",
    "Score",
    "Soil_Comp",
    "Soil_Hydro_Group",
    "ElevChange",
    "NumberofSLs",
    "age_years",
    "breaks_so_far",  #c
    "months_since_last"
]

X = snap_df[feature_cols]

cat_features = [
    "material",
    "Soil_Comp",
    "Soil_Hydro_Group"
]

num_features = [
    c for c in feature_cols if c not in cat_features
]
Xmat = snap_df[num_features + cat_features]
y = snap_df["Y"]

train = snap_df["snapshot"] < "2022-01-01"
X_train = Xmat[train]
X_test  = Xmat[~train]
y_train = y[train]
y_test  = y[~train]

#in case of missing values
num_imp = SimpleImputer(strategy="median")
X_train_num = num_imp.fit_transform(X_train[num_features])
X_test_num  = num_imp.transform(X_test[num_features])

#one hot encoding
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = ohe.fit_transform(X_train[cat_features])
X_test_cat  = ohe.transform(X_test[cat_features])

#train test sets
X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num,  X_test_cat])

LASSO

In [20]:
# Parameter tuning
from sklearn.model_selection import validation_curve

Cs = np.logspace(-4, 1, 6)

X_tune, _, y_tune, _ = train_test_split(
    X_train_final,
    y_train,
    train_size=100_000,
    stratify=y_train,
)

model = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=5000,
    class_weight="balanced",
    tol=0.001,
)

train_scores, val_scores = validation_curve(
    model,
    X_tune,
    y_tune,
    param_name="C",
    param_range=Cs,
    cv=5,
    scoring="average_precision",
    n_jobs=-1
)

In [21]:
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

plt.plot(Cs, train_scores.mean(axis=1), label="Train-fold Average Precision")
plt.plot(Cs, val_scores.mean(axis=1), label="Cross-validation Average Precision")
plt.xscale("log")
plt.xlabel("C")
plt.ylabel("Average Precision")
plt.legend()
plt.show()

In [ ]:
#this takes too long
'''from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 2, 5],
    'tol': [0.001, 0.01, 0.1],
}


lasso_model = LogisticRegression(
    penalty="l1",
    max_iter = 5000,
    solver="saga",
    class_weight="balanced",
)

grid_search = GridSearchCV(
    estimator=lasso_model,
    param_grid=param_grid,
    scoring='average_precision',
    cv=3,
    verbose=1
)

grid_search.fit(X_tune, y_tune)

print(f"Best Score: {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")
'''

Fitting 3 folds for each of 18 candidates, totalling 54 fits


KeyboardInterrupt: 

In [22]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled  = scaler.transform(X_test_final)

lasso_logit = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=5000,
    C=0.01,
    class_weight="balanced",
    tol=0.001
)

In [23]:
lasso_logit.fit(X_train_scaled, y_train)

LogisticRegression(C=0.01, class_weight='balanced', max_iter=5000, penalty='l1',
                   solver='saga', tol=0.001)

In [24]:
print(f"Features used: {np.sum(lasso_logit.coef_ != 0)}")

Features used: 72


In [25]:
p_test_lasso = lasso_logit.predict_proba(X_test_scaled)[:, 1]

In [26]:
X_train_final.shape

(1263600, 73)

XGBoost

In [76]:
#parameter tuning
depths = [3, 5, 7, 10, 13, 16, 20]

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=300,
    tree_method="hist",
    n_jobs=-1,
    gamma = 1,
    reg_lambda = 10,
    subsample=0.8,
    colsample_bytree=0.8
)

train_scores, val_scores = validation_curve(
    xgb,
    X_tune,
    y_tune,
    param_name="max_depth",
    param_range=depths,
    cv=3,
    scoring="average_precision",
    n_jobs=1
)

train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)


In [77]:
plt.figure(figsize=(8,5))
plt.plot(depths, train_mean, marker="o", label="Train Average Precision")
plt.plot(depths, val_mean, marker="o", label="Cross-validation Average Precision")
plt.xlabel("max_depth")
plt.ylabel("Average precision")
plt.legend()
plt.grid(True)
plt.show()

In [65]:
np.arange(0, 1.1, 0.1)

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

In [94]:
lrs = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=300,
    tree_method="hist",
    n_jobs=-1,
    max_depth = 7,
    gamma = 1,
    reg_lambda = 10,
    subsample=0.8,
    colsample_bytree=0.8
)

train_scores, val_scores = validation_curve(
    xgb,
    X_tune,
    y_tune,
    param_name="learning_rate",
    param_range=lrs,
    cv=3,
    scoring="average_precision",
    n_jobs=1
)

train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)

In [79]:
plt.figure(figsize=(8,5))
plt.plot(lrs, train_mean, marker="o", label="Train Average Precision")
plt.plot(lrs, val_mean, marker="o", label="Cross-validation Average Precision")
plt.xlabel("learning_rate")
plt.ylabel("Average precision")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#ended up finding a better way to do this
from sklearn.model_selection import GridSearchCV

param_grid = {
    'learning_rate': [0.2, 0.4, 0.6, 0.8, 1], #dont need to test really high (max 0.01 starting from 0.001)
    'max_depth': [3, 5, 7, 10, 12], #most important learning_rate and max_depth also 999
    'subsample': [0.8], #start with 1
    'colsample_bytree': [0.8], # start with 1
    'gamma': [0, 1, 5]
}


xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=500,
    tree_method="hist",
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='average_precision',
    cv=3,
    verbose=1
)

grid_search.fit(X_tune, y_tune)

print(f"Best Score: {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")
#model.get_score()
#for RF, feature selection in scikit-learn (model.feature_importances_)


Fitting 3 folds for each of 75 candidates, totalling 225 fits
Best Score: 0.7006075095009724
Best Params: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.2, 'max_depth': 12, 'subsample': 0.8}


In [97]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=1000,
    tree_method="hist",
    n_jobs=-1,
    max_depth = 12,
    gamma = 0,
    reg_lambda = 10,
    subsample=0.8,
    colsample_bytree=0.8,
    learning_rate = 0.2
)

In [98]:
xgb.fit(X_train_final, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=0, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=12, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=-1,
              num_parallel_tree=None, random_state=None, ...)

In [99]:
p_test_xgb = xgb.predict_proba(X_test_final)[:, 1]

random forest

In [109]:
from sklearn.ensemble import RandomForestClassifier

In [110]:
param_grid = {
    'max_depth': [5, 7, 10, 12],
    'min_samples_leaf': [5, 10, 12],
}


rf_model = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
)

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='average_precision',
    cv=3,
    verbose=1
)

grid_search.fit(X_tune, y_tune)

print(f"Best Score: {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Score: 0.29171549145752285
Best Params: {'max_depth': 12, 'min_samples_leaf': 5}


In [116]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
)

In [117]:
rf.fit(X_train_final, y_train)

RandomForestClassifier(class_weight='balanced', min_samples_leaf=10,
                       n_estimators=300, n_jobs=-1)

In [118]:
p_test_rf = rf.predict_proba(X_test_final)[:, 1]

evaluation

In [119]:
print("LASSO ROC AUC:", roc_auc_score(y_test, p_test_lasso))
print("LASSO PR AUC :", average_precision_score(y_test, p_test_lasso))

print("XGB ROC AUC:", roc_auc_score(y_test, p_test_xgb))
print("XGB PR AUC :", average_precision_score(y_test, p_test_xgb))

print("RF ROC AUC:", roc_auc_score(y_test, p_test_rf))
print("RF PR AUC :", average_precision_score(y_test, p_test_rf))

LASSO ROC AUC: 0.9086675471764519
LASSO PR AUC : 0.18569111436890923
XGB ROC AUC: 0.9431080180693618
XGB PR AUC : 0.3598769899076369
RF ROC AUC: 0.9269440642483453
RF PR AUC : 0.41333821648268754


final prediction for today

In [121]:
today = pd.Timestamp("2026-02-01")

rows_today = []

for _, pipe in pipes.iterrows():

    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    if pd.isna(install_year):
        continue

    install_year = int(install_year)

    if today.year < install_year:
        continue

    pipe_breaks = breaks.loc[
        breaks["pipe_tag"] == tag,
        "break_date"
    ].sort_values()

    past = pipe_breaks[pipe_breaks < today]

    if len(past) == 0:
        months_since_last = np.nan
        breaks_so_far = 0
    else:
        last_break = past.max()
        months_since_last = (today - last_break).days / 30.44
        breaks_so_far = len(past)

    rows_today.append({
        "tag": tag,
        "diameter": pipe["diameter"],
        "material": pipe["material"],
        "polywrap": pipe["polywrap"],
        "yr_inst": install_year,
        "length": pipe["length"],
        "NumberOfMa": pipe["NumberOfMa"],
        "Shape_Length": pipe["Shape_Length"],
        "Score": pipe["Score"],
        "Soil_Comp": pipe["Soil_Comp"],
        "Soil_Hydro_Group": pipe["Soil_Hydro_Group"],
        "ElevChange": pipe["ElevChange"],
        "NumberofSLs": pipe["NumberofSLs"],
        "age_years": today.year - install_year,
        "breaks_so_far": breaks_so_far,
        "months_since_last": months_since_last
    })

today_df = pd.DataFrame(rows_today)

In [122]:
X_today_num = num_imp.transform(today_df[num_features])
X_today_cat = ohe.transform(today_df[cat_features])
X_today_final = np.hstack([X_today_num, X_today_cat])
X_today_scaled = scaler.transform(X_today_final)

In [123]:
today_df["prob_lasso"] = lasso_logit.predict_proba(X_today_scaled)[:, 1]
today_df["prob_xgb"] = xgb.predict_proba(X_today_final)[:, 1]
today_df["prob_rf"] = rf.predict_proba(X_today_final)[:, 1]

In [125]:
#highest risk pipes lasso
today_df.sort_values("prob_lasso", ascending=False).head(5)

,tag,diameter,material,polywrap,yr_inst,length,NumberOfMa,Shape_Length,Score,Soil_Comp,Soil_Hydro_Group,ElevChange,NumberofSLs,age_years,breaks_so_far,months_since_last,prob_lasso,prob_xgb,prob_rf
9588,wp13214,6.0,CI,0.0,1959,423,6,423.263714,1,CaD,Group C,4.870819,7,67,0,NaN,1.0,0.000515,0.038246
4681,t4570,6.0,CI,0.0,1953,1058,7,1058.166431,1,CtB,Group B,5.793655,2,73,2,11.892247,1.0,0.000189,0.051626
10944,p3844,6.0,CI,0.0,1969,672,7,672.431585,1,CtB,Group B,0.004501,14,57,2,14.914586,1.0,0.077180,0.527170
2260,p3760,4.0,DI,0.0,1987,359,5,359.292162,3,CtC,Group B,0.349106,4,39,0,NaN,1.0,0.015495,0.100883
2467,u4621,6.0,CI,0.0,1957,597,8,597.374772,4,CtB,Group B,3.780563,6,69,4,38.042050,1.0,0.583649,0.557846


In [126]:
#highest risk pipes xgboost
today_df.sort_values("prob_xgb", ascending=False).head(5)

,tag,diameter,material,polywrap,yr_inst,length,NumberOfMa,Shape_Length,Score,Soil_Comp,Soil_Hydro_Group,ElevChange,NumberofSLs,age_years,breaks_so_far,months_since_last,prob_lasso,prob_xgb,prob_rf
6146,s46260,12.0,CI,0.0,1959,294,2,294.416961,2,CtC,Group B,1.707397,0,67,2,60.939553,0.148352,0.967098,0.397252
121,q5217,6.0,CI,0.0,1959,231,2,231.388472,2,CtB,Group B,1.882950,1,67,1,65.111695,0.944439,0.958704,0.414452
12542,q45226,8.0,CIL,0.0,1890,58,2,58.306845,2,CtC,Group B,0.577744,0,136,2,78.153745,0.123858,0.949273,0.395652
1886,v45104,4.0,CI,0.0,1964,212,2,211.891801,2,CtC,Group B,1.767975,1,62,1,121.320631,0.950587,0.935223,0.123286
6395,wp734,8.0,DI,0.0,1993,460,3,459.597400,3,CrC,Group C,1.772369,7,33,3,80.781866,0.210483,0.930189,0.419951


In [127]:
#highest risk pipes random forest
today_df.sort_values("prob_rf", ascending=False).head(5)

,tag,diameter,material,polywrap,yr_inst,length,NumberOfMa,Shape_Length,Score,Soil_Comp,Soil_Hydro_Group,ElevChange,NumberofSLs,age_years,breaks_so_far,months_since_last,prob_lasso,prob_xgb,prob_rf
3003,q5414,6.0,CI,0.0,1965,590,3,589.754846,2,BdB,Group C/D,0.281097,7,61,3,37.089356,0.151620,0.805097,0.751928
5020,wp5808,20.0,CI,0.0,1966,1194,0,1193.842907,3,CrC,Group C/D,7.525696,6,60,1,35.709593,0.002317,0.044237,0.720744
6041,r4479,6.0,CIL,0.0,1927,506,3,505.802708,2,CtB,Group B,0.426270,12,99,3,30.223390,0.140488,0.908308,0.715487
7644,w4220,20.0,CI,0.0,1966,1493,0,1492.767243,3,CrC,Group C/D,8.925140,5,60,1,36.399474,0.003114,0.073941,0.710659
6421,p4683,6.0,CIL,0.0,1910,418,3,417.649665,2,CtB,Group B,6.566284,11,116,2,33.541393,0.925092,0.002545,0.698910


In [128]:
#make it a csv
output = today_df[[
    "tag",
    "prob_lasso",
    "prob_xgb",
    "prob_rf"
]].sort_values("prob_xgb", ascending=False)

output.to_csv("pipe_break_risk_pr2.csv", index=False)